In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import ipywidgets as widgets
from IPython.display import display

In [ ]:
import re
default = "Seleccionar una opcion"
path = "D:/"
files = [f for f in os.listdir(path) if f.endswith('.csv')]
files.insert(0, default)
file_selector = widgets.Dropdown(options=files, description='CSV File:', default=default)
x_min_input = widgets.IntText(value=0, description='X min')
x_max_input = widgets.IntText(value=100, description='X max')
apply_button = widgets.Button(description='Apply X Range')
display(file_selector, x_min_input, x_max_input, apply_button)

def plot_csv(change):
    from IPython.display import clear_output
    if change['type'] == 'change' and change['name'] == 'value' and change['new'] != default:
        clear_output(wait=True)
        display(file_selector, x_min_input, x_max_input, apply_button)
        df = pd.read_csv(path + change['new'], sep=";")
        df["Resistance"] = df["Voltage"] / df["Current"] * 1000
        display(df.head())
        x_min_input.value = 0
        x_max_input.value = len(df)-1
        x_max_input.max = len(df)-1
        # Extract title from filename using regex
        def update_plot(b):
            clear_output(wait=True)
            display(file_selector, x_min_input, x_max_input, apply_button)
            display(df.head())
            x_min = max(0, min(x_min_input.value, len(df)-1))
            x_max = max(x_min, min(x_max_input.value, len(df)-1))
            fig = make_subplots(rows=2, cols=2)
            fig.add_trace(go.Scatter(x=df.index[x_min:x_max+1], y=df["Current"][x_min:x_max+1], name="Current"), row=1, col=1)
            # fig.add_trace(go.Scatter(x=df.index[x_min:x_max+1], y=df["Resistance"][x_min:x_max+1], name="Resistance"), row=1, col=2)
            fig.add_trace(go.Scatter(x=df.index[x_min:x_max+1], y=df["Error"][x_min:x_max+1], name="Error"), row=1, col=2)
            fig.add_trace(go.Scatter(x=df.index[x_min:x_max+1], y=df["Resistance"][x_min:x_max+1], name="Resistance"), row=2, col=1)
            fig.add_trace(go.Scatter(x=df.index[x_min:x_max+1], y=df["R_Target"][x_min:x_max+1], name="R_Target"), row=2, col=2)
            fig_title = change["new"][:-4].split("-")[2].split("__")
            target = fig_title[0][:3]
            Kp, Ki, Kd= fig_title[1].split("_")
            fig.update_layout(title=f"Resistance {target} - Kp: {Kp}, Ki: {Ki}, Kd: {Kd}")
            fig.show()
        apply_button.on_click(update_plot)

file_selector.observe(plot_csv)

In [3]:
from pandas.errors import EmptyDataError
archivo_salida = "datos.csv"
columns = ["Voltage", "Current", "PWM Value", "Error", "Integral", "Derivative", "R_Target"]
try:
    df = pd.read_csv(archivo_salida)
except EmptyDataError:
    print(f"El archivo {archivo_salida} está vacío o no existe.")
    df = pd.DataFrame()
df.columns = columns
df["error%"] = (df["Error"] / (1000 * df["Voltage"] / df["R_Target"])) * 100
df["Resistance"] = 1000* df["Voltage"] / df["Current"]
df["R_error%"] = (df["Resistance"] - df["R_Target"]) / df["R_Target"] * 100

fig = make_subplots(rows=3, cols=2)
fig.add_trace(go.Scatter(x=df.index, y=df["Current"], name="Current"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["Error"], name="Error"), row=1, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df["Derivative"], name="Derivative"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["Integral"], name="Integral"), row=2, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df["R_Target"], name="R_Target"), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["R_error%"], name="R_error%"), row=3, col=2)

fig.update_layout(
    title="Carga Electrónica - Analisis de rendimiento",
    legend_title="Datos",
    template="plotly_dark",
)
# fig.add_trace(go.Scatter(x=df.index, y=df["error%"], name="error%"), row=3, col=2)